<a href="https://colab.research.google.com/github/Kushal2409/AI-LAB/blob/main/Lab_9_1WN24CS143_ForwardReasoning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import re

# ==========================================
# 1. PARSE FOL EXPRESSIONS
# ==========================================

def parse_predicate(expression):

    expression = expression.strip()

    match = re.fullmatch(
        r'([A-Za-z_][A-Za-z0-9_]*)\s*\((.*)\)',
        expression
    )

    if not match:
        raise ValueError("Invalid predicate: " + expression)

    name = match.group(1)
    args = match.group(2).strip()

    if not args:
        return (name,)

    arguments = tuple(
        arg.strip() for arg in args.split(",")
    )

    return (name,) + arguments


def parse_rule(expression):

    parts = re.split(r'\s*(?:->|=>)\s*', expression)

    if len(parts) != 2:
        raise ValueError("Invalid rule format")

    premises = re.split(r'\s*(?:\^|&)\s*', parts[0])

    premises = [
        parse_predicate(p) for p in premises
    ]

    conclusion = parse_predicate(parts[1])

    return premises, conclusion


# ==========================================
# 2. VARIABLE CHECK
# ==========================================

def is_variable(term):
    return isinstance(term, str) and term.startswith("?")


# ==========================================
# 3. SUBSTITUTION
# ==========================================

def subst(theta, expr):

    if is_variable(expr):

        while expr in theta:
            value = theta[expr]

            if value == expr:
                break

            expr = value

        return expr

    if isinstance(expr, tuple):
        return tuple(subst(theta, x) for x in expr)

    return expr


# ==========================================
# 4. UNIFICATION
# ==========================================

def unify(x, y, theta=None):

    if theta is None:
        theta = {}

    if theta is False:
        return False

    x = subst(theta, x)
    y = subst(theta, y)

    if x == y:
        return theta

    if is_variable(x):
        return unify_var(x, y, theta)

    if is_variable(y):
        return unify_var(y, x, theta)

    if isinstance(x, tuple) and isinstance(y, tuple):

        if len(x) != len(y):
            return False

        for a, b in zip(x, y):

            theta = unify(a, b, theta)

            if theta is False:
                return False

        return theta

    return False


def unify_var(var, x, theta):

    if var == x:
        return theta

    if var in theta:
        return unify(theta[var], x, theta)

    if is_variable(x) and x in theta:
        return unify(var, theta[x], theta)

    if isinstance(x, tuple) and var in get_variables(x):
        return False

    new_theta = theta.copy()
    new_theta[var] = x

    return new_theta


def get_variables(expr):

    if is_variable(expr):
        return {expr}

    if isinstance(expr, tuple):

        result = set()

        for item in expr:
            result.update(get_variables(item))

        return result

    return set()


# ==========================================
# 5. STANDARDIZE VARIABLES
# ==========================================

var_counter = 0


def standardize_variables(rule):

    global var_counter
    var_counter += 1

    premises, conclusion = rule
    mapping = {}

    def rename(term):

        if is_variable(term):

            if term not in mapping:
                mapping[term] = term + "_" + str(var_counter)

            return mapping[term]

        if isinstance(term, tuple):
            return tuple(rename(x) for x in term)

        return term

    return (
        [rename(p) for p in premises],
        rename(conclusion)
    )


# ==========================================
# 6. FIND SUBSTITUTIONS
# ==========================================

def find_substitutions(premises, facts, theta=None):

    if theta is None:
        theta = {}

    if not premises:
        yield theta
        return

    first = premises[0]
    rest = premises[1:]

    for fact in facts:

        new_theta = unify(first, fact, theta)

        if new_theta is not False:

            yield from find_substitutions(
                rest, facts, new_theta
            )


# ==========================================
# 7. FORWARD CHAINING
# ==========================================

def fol_fc_ask(facts, rules, query):

    facts = list(facts)

    while True:

        new_facts = []

        for rule in rules:

            premises, conclusion = standardize_variables(rule)

            for theta in find_substitutions(premises, facts):

                result = subst(theta, conclusion)

                if result not in facts and result not in new_facts:

                    new_facts.append(result)

                    answer = unify(result, query)

                    if answer is not False:
                        return answer

        if not new_facts:
            return False

        facts.extend(new_facts)


# ==========================================
# 8. MAIN PROGRAM
# ==========================================

if __name__ == "__main__":

    print("\n===== FOL FORWARD CHAINING =====")

    # INPUT KNOWLEDGE BASE

    facts = []

    n = int(input("\nNumber of facts: "))

    print("Enter facts (Example: American(Robert))")

    for i in range(n):

        facts.append(
            parse_predicate(input(f"Fact {i+1}: "))
        )

    # INPUT RULES

    rules = []

    n = int(input("\nNumber of rules: "))

    print("\nEnter rules using:")
    print("^ for AND")
    print("-> for IMPLIES")
    print("Use ?x, ?y, ?z for variables")

    for i in range(n):

        expression = input(f"Rule {i+1}: ")

        rules.append(parse_rule(expression))

    # INPUT QUERY

    query = parse_predicate(
        input("\nEnter query: ")
    )

    # INFERENCE

    result = fol_fc_ask(facts, rules, query)

    print("\n========== RESULT ==========")

    if result is not False:

        print("QUERY IS PROVED!")
        print("Substitution:", result)

    else:

        print("QUERY CANNOT BE PROVED.")



===== FOL FORWARD CHAINING =====

Number of facts: 3 
Enter facts (Example: American(Robert))
Fact 1: Barks(Tommy)
Fact 2: FourLegs(Tommy)
Fact 3: SharpTeeth(Tommy)

Number of rules: 5

Enter rules using:
^ for AND
-> for IMPLIES
Use ?x, ?y, ?z for variables
Rule 1: Barks(?x) ^ FourLegs(?x) -> Dog(?x)
Rule 2: Feather(?x) ^ LaysEggs(?x) -> Bird(?x)
Rule 3: Dog(?x) -> Mammal(?x)
Rule 4: Mammal(?x) ^ SharpTeeth(?x) -> Carnivore(?x)
Rule 5: Bird(?x) -> Fly(?x)

Enter query: Carnivore(Tommy)

========== RESULT ==========
QUERY IS PROVED!
Substitution: {}
